In [0]:
from pyspark.sql.functions import *

# Read both tables individually
df = spark.read.table("fq_dev_pnl_catalog.bronze.gl_report")


# df = spark.read.table('default.postman_response_january')
df_exploded = df.select(explode('results').alias('result')).select('result.*').filter(col('year') == 2026).filter((col('month') == 'FEB') | (col('month') == 'JAN'))



# Load brand allocation data
df_brand_allocation_cost = spark.read.table('fq_dev_catalog.financial_pnl.brand_ho_allocation_cost')

df_brand_allocation_cost = df_brand_allocation_cost.withColumn("brand_ho_allocation_cost", col("brand_ho_allocation_cost").cast("string"))

# Based on the schema, transform brand allocation to match exploded format
# Use when() to handle "-" values and convert to NULL
df_brand_ho_rows = df_brand_allocation_cost.select(
    lit("73219").cast("string").alias("accountNo"),
    lit("73219 - Brand HO allocation cost").cast("string").alias("accountName"),
    col("netsuite_location_name").cast("string").alias("location"),
    # ✅ Use when() to convert "-" to NULL, then cast to double
    when(col("brand_ho_allocation_cost") == "-", lit(None))
        .otherwise(col("brand_ho_allocation_cost"))
        .try_cast("double")
        .alias("amount"),
    col("month").cast("string").alias("month"),
    col("year").cast("string").alias("year")
).withColumn("accounttype", lit("Expense").cast("string"))

# Filter out rows where amount is NULL (from malformed "-" values)
df_brand_ho_rows = df_brand_ho_rows.filter(col("amount").isNotNull())

# Add any other columns from df_exploded as nulls
target_columns = df_exploded.columns
for column in target_columns:
    if column not in df_brand_ho_rows.columns:
        # Get the data type from df_exploded
        col_type = [f.dataType for f in df_exploded.schema if f.name == column][0]
        df_brand_ho_rows = df_brand_ho_rows.withColumn(column, lit(None).cast(col_type))

# Reorder columns to match df_exploded
df_brand_ho_rows = df_brand_ho_rows.select(df_exploded.columns)
df_brand_ho_rows.display()

# Now union 
df_exploded = df_exploded.union(df_brand_ho_rows)

print(f"\n✅ Union successful! Total rows: {df_exploded.count()}")
# df_exploded.display()

In [0]:
df_brand_ho_rows.write.mode("overwrite").option('overwriteSchema', True).saveAsTable("fq_dev_pnl_catalog.bronze.brand_ho_allocation_cost")

In [0]:
# %skip 
from pyspark.sql.functions import *

# Read both tables individually
df = spark.read.table("fq_dev_pnl_catalog.bronze.gl_report")

df_exploded = df.select(explode('results').alias('result')).select('result.*').filter(col('year') == 2026).filter((col('month') == 'FEB'))

# ✅ Reverse sign when accountType == 'Income'
df_exploded = df_exploded.withColumn(
    "amount",
    when(col("accountType") == "Income", -col("amount"))
    .when(col("accountName") == "75704 Talabat- Commission Discount", -col("amount"))
    .when(col("accountName") == "75713 Noon- Commission Discount", -col("amount"))
    .otherwise(col("amount"))
).withColumn(
    "year", col('year').cast('int')
)

# Load brand allocation data
df_brand_ho_rows = spark.read.table('fq_dev_pnl_catalog.bronze.brand_ho_allocation_cost')

df_exploded_keys = df_exploded.select(
    col("year"), 
    col("month"), 
    col("location").alias("netsuite_location_name")
).distinct()


df_brand_ho_rows_filtered = df_brand_ho_rows.join(
    df_exploded_keys,
    (df_brand_ho_rows["year"] == df_exploded_keys["year"]) &
    (df_brand_ho_rows["month"] == df_exploded_keys["month"]) &
    (df_brand_ho_rows["location"] == df_exploded_keys["netsuite_location_name"]),
    "inner"
).select(df_brand_ho_rows["*"])

# Perform union with filtered data
df_exploded = df_exploded.union(df_brand_ho_rows_filtered)
df_exploded.display()
print(f"\n✅ Union successful! Total rows: {df_exploded.count()}")

In [0]:
# %skip 
from pyspark.sql.functions import *

# Read both tables individually
df = spark.read.table("fq_dev_pnl_catalog.bronze.gl_report")

df_exploded = df.select(explode('results').alias('result')).select('result.*').filter(col('year') == 2026).filter((col('month') == 'FEB') | (col('month') == 'JAN'))

# ✅ Reverse sign when accountType == 'Income'
df_exploded = df_exploded.withColumn(
    "amount",
    when(col("accountType") == "Income", -col("amount"))
    .when(col("accountName") == "75704 Talabat- Commission Discount", -col("amount"))
    .when(col("accountName") == "75713 Noon- Commission Discount", -col("amount"))
    .otherwise(col("amount"))
).withColumn(
    "year", col('year').cast('int')
)

# Load brand allocation data
df_brand_allocation_cost = spark.read.table('fq_dev_catalog.financial_pnl.brand_ho_allocation_cost')

df_brand_allocation_cost = df_brand_allocation_cost.withColumn("brand_ho_allocation_cost", col("brand_ho_allocation_cost").cast("string"))

df_brand_ho_rows = df_brand_allocation_cost.select(
    lit("73219").cast("string").alias("accountNo"),
    lit("73219 - Brand HO allocation cost").cast("string").alias("accountName"),
    col("netsuite_location_name").cast("string").alias("location"),
    when(col("brand_ho_allocation_cost") == "-", lit(None))
        .otherwise(col("brand_ho_allocation_cost"))
        .try_cast("double")
        .alias("amount"),
    col("month").cast("string").alias("month"),
    col("year").cast("string").alias("year")
).withColumn("accounttype", lit("Expense").cast("string"))

df_brand_ho_rows = df_brand_ho_rows.filter(col("amount").isNotNull())

# ═══════════════════════════════════════════════════════════════
# 🔍 VALIDATION: Check matching keys before union
# ═══════════════════════════════════════════════════════════════

# Convert year to string in df_exploded for comparison
df_exploded_keys = df_exploded.select(
    col("year").cast("string").alias("year"), 
    col("month"), 
    col("location").alias("netsuite_location_name")
).distinct()

df_brand_keys = df_brand_ho_rows.select(
    col("year"), 
    col("month"), 
    col("location").alias("netsuite_location_name")
).distinct()

# Find matching keys
matching_keys = df_exploded_keys.intersect(df_brand_keys)
matching_count = matching_keys.count()

# Find keys only in df_exploded
only_in_exploded = df_exploded_keys.subtract(df_brand_keys)
exploded_only_count = only_in_exploded.count()

# Find keys only in df_brand_ho_rows
only_in_brand = df_brand_keys.subtract(df_exploded_keys)
brand_only_count = only_in_brand.count()

# Print validation summary
print("=" * 70)
print("🔍 VALIDATION SUMMARY: Key Matching Check")
print("=" * 70)
print(f"✅ Matching combinations (year, month, location): {matching_count}")
print(f"⚠️  Only in df_exploded: {exploded_only_count}")
print(f"⚠️  Only in df_brand_ho_rows: {brand_only_count}")
print("=" * 70)

if exploded_only_count > 0:
    print("\n📋 Locations in df_exploded NOT in brand allocation:")
    only_in_exploded.orderBy("year", "month", "netsuite_location_name").show(20, truncate=False)

if brand_only_count > 0:
    print("\n📋 Locations in brand allocation NOT in df_exploded:")
    only_in_brand.orderBy("year", "month", "netsuite_location_name").show(20, truncate=False)

if matching_count > 0:
    print("\n✅ Sample of matching locations:")
    matching_keys.orderBy("year", "month", "netsuite_location_name").show(10, truncate=False)

# ═══════════════════════════════════════════════════════════════
# Decision point: Filter to only matching keys or proceed with all?
# ═══════════════════════════════════════════════════════════════

# Option 1: Only union records with matching keys (RECOMMENDED)
print("\n🔧 Filtering df_brand_ho_rows to only matching keys...")
df_brand_ho_rows_filtered = df_brand_ho_rows.join(
    matching_keys,
    (df_brand_ho_rows["year"] == matching_keys["year"]) &
    (df_brand_ho_rows["month"] == matching_keys["month"]) &
    (df_brand_ho_rows["location"] == matching_keys["netsuite_location_name"]),
    "inner"
).select(df_brand_ho_rows["*"])

print(f"Filtered brand rows: {df_brand_ho_rows_filtered.count()} (down from {df_brand_ho_rows.count()})")

# Add missing columns and align schema
target_columns = df_exploded.columns
for column in target_columns:
    if column not in df_brand_ho_rows_filtered.columns:
        col_type = [f.dataType for f in df_exploded.schema if f.name == column][0]
        df_brand_ho_rows_filtered = df_brand_ho_rows_filtered.withColumn(column, lit(None).cast(col_type))

df_brand_ho_rows_filtered = df_brand_ho_rows_filtered.select(df_exploded.columns)

# Perform union with filtered data
df_exploded = df_exploded.union(df_brand_ho_rows_filtered)

print(f"\n✅ Union successful! Total rows: {df_exploded.count()}")